# Setting your Jupyter AI API keys and endpoints

To avoid the hassle of having to set your Jupyter AI API key/endpoint every time you start a new session, we've provided the following cells that will populate ~/.profile with your personal API keys and endpoints.

A note about security: ~/.profile is saved to your individual EBS mounted storage drive. It is not visible anyone but you. The only way to expose your API keys to others using HelioCloud would be to save them to the shared drive at ```scratch_space/```. We would also advise you to add .profile to your .gitignore if you are saving your whole user file system to a public Git repo.

**Each time you change the API variables in ~/.profile, you will need to make sure to restart your Jupyter server to source ~/.profile again.**

# Set API Keys

In [ ]:
from pathlib import Path

import ipywidgets as widgets
from IPython.display import display, clear_output

CHOICES = [
    ("Gemini (Google AI Studio) → GOOGLE_API_KEY", "GOOGLE_API_KEY"),
    ("OpenAI → OPENAI_API_KEY", "OPENAI_API_KEY"),
    ("Anthropic → ANTHROPIC_API_KEY", "ANTHROPIC_API_KEY"),
    ("Cohere → COHERE_API_KEY", "COHERE_API_KEY"),
    ("MistralAI → MISTRAL_API_KEY", "MISTRAL_API_KEY"),
    ("NVIDIA → NVIDIA_API_KEY", "NVIDIA_API_KEY"),
    ("Hugging Face Hub → HUGGINGFACEHUB_API_TOKEN", "HUGGINGFACEHUB_API_TOKEN"),
    ("AI21 → AI21_API_KEY", "AI21_API_KEY"),
    ("Other… (enter custom env var name)", "__OTHER__"),
]

profile_path = Path.home() / ".profile"
profile_path.touch(exist_ok=True)

dropdown = widgets.Dropdown(
    options=CHOICES,
    value="GOOGLE_API_KEY",
    description="Provider:",
    layout=widgets.Layout(width="700px"),
)

custom_name = widgets.Text(
    value="",
    placeholder='e.g., MY_PROVIDER_API_KEY',
    description="Env var:",
    layout=widgets.Layout(width="700px"),
)
custom_name.layout.display = "none"

secret_box = widgets.Password(
    value="",
    placeholder="Paste API key here (masked)",
    description="API key:",
    layout=widgets.Layout(width="700px"),
)

save_btn = widgets.Button(description="Save to ~/.profile", button_style="primary")
out = widgets.Output()

def upsert_export_line(path: Path, key: str, value: str) -> None:
    header = "# Jupyter-ai API keys"

    text = path.read_text()
    lines = text.splitlines()

    # Locate header section
    if header in lines:
        start = lines.index(header) + 1
    else:
        # create section at end
        if lines and lines[-1].strip() != "":
            lines.append("")
        lines.append(header)
        start = len(lines)

    # Find where section ends (next blank line or EOF)
    end = start
    while end < len(lines) and lines[end].startswith("export "):
        end += 1

    section = lines[start:end]

    updated = False
    for i, line in enumerate(section):
        if line.startswith(f"export {key}="):
            section[i] = f'export {key}="{value}"'
            updated = True

    if not updated:
        section.append(f'export {key}="{value}"')

    lines[start:end] = section

    path.write_text("\n".join(lines) + "\n")

def on_dropdown_change(change):
    if change["name"] == "value":
        if change["new"] == "__OTHER__":
            custom_name.layout.display = ""
        else:
            custom_name.layout.display = "none"
            custom_name.value = ""

dropdown.observe(on_dropdown_change)

def on_save(_):
    with out:
        clear_output()

        env_name = custom_name.value.strip() if dropdown.value == "__OTHER__" else dropdown.value
        if not env_name:
            print("❌ Please enter an environment variable name.")
            return

        secret = secret_box.value
        if not secret.strip():
            print("❌ Please enter a non-empty API key.")
            return

        upsert_export_line(profile_path, env_name, secret)

        # Clear the UI value so the secret isn't left sitting in widget state
        secret_box.value = ""

        print(f"✅ Saved {env_name} to {profile_path} (overwritten if it existed).")
        print("⚠️ Restart your Jupyter server so a new session picks up ~/.profile.")

save_btn.on_click(on_save)

display(dropdown, custom_name, secret_box, save_btn, out)

# Set API Endpoints

You may need to route API calls to a specific URL endpoint. You can use the code below to add API endpoint URLs to ~/.profile as well.

In [ ]:
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output

profile_path = Path.home() / ".profile"
profile_path.touch(exist_ok=True)

CHOICES = [
    ("OpenAI base URL (OPENAI_BASE_URL)", "OPENAI_BASE_URL"),
    ("Anthropic base URL (ANTHROPIC_BASE_URL)", "ANTHROPIC_BASE_URL"),
    ("Other…", "__OTHER__"),
]

dropdown = widgets.Dropdown(
    options=CHOICES,
    value="OPENAI_BASE_URL",
    description="Endpoint:",
    layout=widgets.Layout(width="700px"),
)

custom_name = widgets.Text(
    value="",
    placeholder="e.g., MY_PROVIDER_BASE_URL",
    description="Env var:",
    layout=widgets.Layout(width="700px"),
)
custom_name.layout.display = "none"

url_box = widgets.Text(
    value="",
    placeholder="https://api.example.com/v1",
    description="URL:",
    layout=widgets.Layout(width="700px"),
)

save_btn = widgets.Button(description="Save to ~/.profile", button_style="primary")
out = widgets.Output()


def upsert_export_line(path: Path, key: str, value: str) -> None:
    header = "# Jupyter-ai API keys"

    text = path.read_text()
    lines = text.splitlines()

    if header in lines:
        start = lines.index(header) + 1
    else:
        if lines and lines[-1].strip() != "":
            lines.append("")
        lines.append(header)
        start = len(lines)

    end = start
    while end < len(lines) and lines[end].startswith("export "):
        end += 1

    section = lines[start:end]

    updated = False
    for i, line in enumerate(section):
        if line.startswith(f"export {key}="):
            section[i] = f'export {key}="{value}"'
            updated = True

    if not updated:
        section.append(f'export {key}="{value}"')

    lines[start:end] = section

    path.write_text("\n".join(lines) + "\n")


def on_dropdown_change(change):
    if change["name"] == "value":
        if change["new"] == "__OTHER__":
            custom_name.layout.display = ""
        else:
            custom_name.layout.display = "none"
            custom_name.value = ""


dropdown.observe(on_dropdown_change)


def on_save(_):
    with out:
        clear_output()

        env_name = custom_name.value.strip() if dropdown.value == "__OTHER__" else dropdown.value
        if not env_name:
            print("❌ Please enter an environment variable name.")
            return

        url = url_box.value.strip()
        if not url:
            print("❌ Please enter a URL.")
            return

        upsert_export_line(profile_path, env_name, url)

        url_box.value = ""

        print(f"✅ Saved {env_name} to {profile_path} (overwritten if it existed).")
        print("⚠️ Restart the Jupyter server so the new environment variable is available.")


save_btn.on_click(on_save)

display(dropdown, custom_name, url_box, save_btn, out)

# Check ~/.profile

If you want to check that everything has updated correctly in ~/.profile, you can run the cell below to see that your (masked) API keys and base URLs have populated.

In [ ]:
from pathlib import Path

profile_path = Path.home() / ".profile"

if not profile_path.exists():
    raise FileNotFoundError(f"No {profile_path} found.")

SECRET_PATTERNS = ["KEY", "TOKEN", "SECRET", "PASSWORD"]

def redact_export_line(line: str, show_last: int = 4) -> str:
    s = line.strip()

    if not s.startswith("export ") or "=" not in s:
        return line

    left, right = s.split("=", 1)   # left: export NAME
    var_name = left.replace("export ", "").strip()
    value = right.strip()

    # Do not redact URL/base endpoint variables
    if "URL" in var_name or "ENDPOINT" in var_name:
        return line

    # Only redact likely secrets
    if not any(p in var_name for p in SECRET_PATTERNS):
        return line

    # Remove quotes if present
    quote = ""
    if (value.startswith('"') and value.endswith('"')) or (value.startswith("'") and value.endswith("'")):
        quote = value[0]
        value = value[1:-1]

    tail = value[-show_last:] if len(value) >= show_last else value
    redacted = "*" * max(0, len(value) - len(tail)) + tail

    if quote:
        return f"{left}={quote}{redacted}{quote}\n"
    else:
        return f"{left}={redacted}\n"


for line in profile_path.read_text(encoding="utf-8").splitlines(keepends=True):
    print(redact_export_line(line), end="")